# 01 — The model boundary: prompts as interfaces, JSON as a contract

**What you'll learn**

- What `litellm.completion` actually returns: `choices`, `message`, `usage`, and the cost hiding in `_hidden_params`
- Why a prompt is an interface — and why prose is not a return type
- How model JSON really arrives (fenced, wrapped in chatter) and how `parse_json_loose` turns it into data or fails loudly
- The one door all later chapters call through: `shoplab.llm.complete`, its `LEDGER`, and `usage_summary()`
- What structured output cannot fix: a model with no data access invents the facts

*Time: ~5 min. Cost: ~$0.001. Cached reruns are free.*

An LLM call is an API call with an unreliable payload. The transport is boring: HTTPS out, JSON back, some latency, a price per token. The payload is not boring — the field you care about is a string the model composed, and nothing about it is guaranteed. Not the shape, not the vocabulary, not the facts. Every system that calls a model inherits this asymmetry, and most agent failures in this course are this asymmetry wearing a costume.

Everything agentic sits on this boundary. The tool loop of chapter 02 is this call inside a `while` loop; the evals of chapter 04 grade what crosses it; the guardrails of chapters 08 and 09 exist because of it. So chapter one stays at the boundary itself: make one call, look hard at what comes back, and build the two pieces everything else leans on — a parser that turns model text into data or raises, and a wrapper that writes down what every call cost.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

With Phoenix running, every call this notebook makes is traced and inspectable in its UI on port 6006. Skip it freely; chapter 03 gives it a proper tour.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## The response is a data structure

One call, no wrappers. The course speaks to every model through LiteLLM, which picks the backend from the provider prefix of the model string ([LiteLLM providers](https://docs.litellm.ai/docs/providers)); the default `MODEL` routes to OpenRouter, which fronts hundreds of models behind one endpoint and one key ([OpenRouter quickstart](https://openrouter.ai/docs/quickstart)).

`messages` is a list because the API is stateless: there is no session on the server, so every call resends the whole conversation. That one fact quietly shapes the course — context windows fill up (chapter 10), and "memory" is something you build, not something you have.

In [ ]:
import litellm

response = litellm.completion(
    model=MODEL,
    messages=[{"role": "user", "content":
               "Reply with one short sentence: what is a restocking fee?"}],
    temperature=TEMPERATURE,
)
print(response.choices[0].message.content)

> **What you should see:** a one-sentence definition, as plain text. The payload sits three attribute hops deep: `response.choices[0].message.content`.

The string is the smallest part of what came back. The `ModelResponse` object carries `choices` (each with a `message` and a `finish_reason` — `stop` means the model chose to end, `length` means your token limit cut it off), `usage` with token counts, and, LiteLLM-specific, `_hidden_params` with the computed dollar cost of the call. Cost is not metadata to skim past; it is the resource every later chapter budgets.

In [ ]:
choice = response.choices[0]
print("type          :", type(response).__name__)
print("finish_reason :", choice.finish_reason)
print("message.role  :", choice.message.role)
print("usage         :", response.usage.prompt_tokens, "prompt +",
      response.usage.completion_tokens, "completion tokens")
print("cost_usd      :", response._hidden_params.get("response_cost"))

> **What you should see:** `finish_reason` is `stop`, both token counts in the low double digits, and `cost_usd` in the hundred-thousandths of a dollar — well under a hundredth of a cent. Rerun the cell: the disk cache answers instantly, and a cost figure is still reported.

## Prose is not a return type

Meet the working object of the whole course: a return ticket at Larkspur Outfitters, the small outdoor-gear shop from chapter 00. A customer bought the Torrent boots, wore them for one evening indoors, and wants a refund. The ticket carries structured fields alongside the complaint, plus a gold label that we will hold back until the end of the chapter.

The obvious first prompt states the job and asks for a decision. Load the ticket, then ask.

In [ ]:
from shoplab import world

ticket = next(t for t in world.load_tickets()["train"]
              if t["ticket_id"] == "TKT-2205")
print(ticket["reason_text"])
print({k: ticket[k] for k in ("requested_action", "item_condition",
                              "days_since_delivery", "evidence_photo")})

In [ ]:
PROMPT_V1 = f"""You run the ops desk at Larkspur Outfitters, an online outdoor-gear shop.
A customer writes about order {ticket["order_id"]}:

{ticket["reason_text"]}

Decide what the customer gets, and say why in a sentence or two."""

reply = litellm.completion(model=MODEL, temperature=TEMPERATURE, max_tokens=200,
                           messages=[{"role": "user", "content": PROMPT_V1}])
print(reply.choices[0].message.content)

> **What you should see:** a defensible-sounding ruling with a short justification — as prose. Note what you cannot do with it: there is no field to branch on, no amount to hand a payments system, and the phrasing can drift between runs and models. A human could act on this reply. A program cannot.

## JSON as a contract

The fix looks obvious: show the model the shape you want back. This is schema-in-prompt — the lightest form of structured output. No tool schemas, no response-format parameters, just an example object and the word "exactly". It works surprisingly often.

In [ ]:
import json

SCHEMA = '{"decision": "...", "policy_id": "...", "refund_usd": 0.0}'

PROMPT_V2 = (PROMPT_V1 + "\n\nAnswer with a single JSON object shaped exactly like\n"
             + SCHEMA + "\nand nothing else.")
raw = litellm.completion(model=MODEL, temperature=TEMPERATURE, max_tokens=200,
                         messages=[{"role": "user", "content": PROMPT_V2}]
                         ).choices[0].message.content
print(raw)
json.loads(raw)

> **What you should see:** a bare JSON object, and `json.loads` accepts it. Now read the values: the decision is a sentence rather than a label, and the policy id and the amount are pure invention — nothing in the prompt could ground them. The shape complied; the content is improvised. Hold that thought — first, the shape itself is flimsier than one clean run suggests.

Ask for anything else alongside the object — reasoning, a caveat — and the model will oblige, in the same string. The next prompt does exactly that, and it is not a contrived failure: "explain, then answer" is how you will *want* to prompt in chapter 05, because reasoning before answering tends to buy accuracy.

In [ ]:
PROMPT_V3 = PROMPT_V2.replace("and nothing else.",
                              "Explain your reasoning first, then give the JSON.")
raw3 = litellm.completion(model=MODEL, temperature=TEMPERATURE, max_tokens=400,
                          messages=[{"role": "user", "content": PROMPT_V3}]
                          ).choices[0].message.content
print(raw3)
try:
    json.loads(raw3)
except json.JSONDecodeError as e:
    print("\njson.loads:", e)

> **What you should see:** a paragraph of reasoning, then the object inside a ``` fence — and `json.loads` dies at character 0. Nothing malfunctioned. The model did what we asked; the naive party here is the parser.

## Parse like you mean it

You could tighten the prompt and hope. The course takes the other route: accept that model output is text shaped by chat habits — markdown fences, helpful chatter — and parse accordingly. Two mechanical steps cover almost everything. Step one: if the reply opens with a code fence, cut the fence lines off.

In [ ]:
def strip_fences(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    return text

print(strip_fences('```json\n{"decision": "deny"}\n```'))

Step two handles chatter. Find the first `{` or `[`, scan forward counting nesting depth, and where depth returns to zero the object ends; slice it out and `json.loads` the slice. Here is the naive version, pointed at the reply that just broke `json.loads`.

In [ ]:
def first_object(text):
    start = text.find("{")
    depth = 0
    for i, ch in enumerate(text[start:], start):
        depth += (ch == "{") - (ch == "}")
        if depth == 0:
            return text[start:i + 1]

json.loads(first_object(raw3))

Three edge cases separate that sketch from something you can trust. A brace inside a JSON string — `"the label said }fragile{"` — fools the depth counter, so real bracket-matching must track string state and escapes. Arrays are legal top-level JSON. And when there is no JSON at all, the function must raise, not return `None`. The version below handles all three, and it is not a copy of the course's parser — it *is* the course's parser. This cell ships verbatim as `shoplab.llm.parse_json_loose`; the build validator byte-compares the notebook cell against the package source, so the two can never drift.

In [ ]:
# >>> shoplab.llm.parse_json_loose
def parse_json_loose(text: str):
    """Parse the first balanced JSON object or array in model output,
    tolerating markdown fences, surrounding prose, and trailing garbage."""
    text = text.strip()
    if text.startswith("```"):                  # drop ```json ... ``` fences
        text = text.split("\n", 1)[1] if "\n" in text else ""
        text = text.rsplit("```", 1)[0]
    starts = [i for i in (text.find("{"), text.find("[")) if i != -1]
    if not starts:
        raise ValueError("no JSON found in model output")
    start, depth, in_string, escaped = min(starts), 0, False, False
    for i, ch in enumerate(text[start:], start):
        if in_string:                           # brackets inside strings do not count
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
        elif ch == '"':
            in_string = True
        elif ch in "{[":
            depth += 1
        elif ch in "}]":
            depth -= 1
            if depth == 0:
                return json.loads(text[start:i + 1])
    raise ValueError("no JSON found in model output")
# <<< shoplab.llm.parse_json_loose

In [ ]:
fixtures = [
    '```json\n{"decision": "deny", "refund_usd": null}\n```',
    'Sure -- the decision: {"decision": "approve_refund"} Let me know!',
    '{"note": "label said }fragile{ on it", "decision": "deny"}',
    "Refund the customer, obviously.",
]
for fx in fixtures:
    try:
        print("parsed:", parse_json_loose(fx))
    except ValueError as e:
        print("ValueError:", e)

> **What you should see:** three dicts, then `ValueError: no JSON found in model output`. The fence is stripped, the chatter ignored, the brace inside a quoted string not miscounted — and the JSON-free reply refuses to parse.

That `ValueError` is a feature — arguably the most important line in the function. At the model boundary, garbage must terminate the operation, not limp onward. A parser that shrugs and returns `None` or `{}` converts a model failure into a silent downstream one: an empty decision recorded, a zero-dollar refund posted, an eval scored against nothing. Raising turns the same failure into a stack trace you see immediately — and in chapter 02 it becomes an error message the model itself gets to read and correct.

## Every call leaves a receipt

This notebook has already spent money nobody counted. That is fine at five calls; it is not fine in chapter 04, where an eval sweep is hundreds of calls, or chapter 07, where an optimizer makes calls to improve other calls. The course rule: every model call goes through one function, and that function writes a ledger row. Build the minimal version first.

In [ ]:
MINI_LEDGER = []

def complete_mini(messages, **kw):
    r = litellm.completion(model=MODEL, temperature=TEMPERATURE,
                           messages=messages, **kw)
    MINI_LEDGER.append({"total_tokens": r.usage.total_tokens,
                        "cost_usd": r._hidden_params.get("response_cost")})
    return r

complete_mini([{"role": "user", "content": "Say OK."}])
MINI_LEDGER

> **What you should see:** one row — a handful of tokens, a cost in the millionths of dollars.

The shipped version, `shoplab.llm.complete`, keeps this shape and records model, split token counts, cost, and wall-clock seconds per row. Two conveniences ride along: `llm(prompt)` for the one-string-in, one-string-out case, and `usage_summary()` to total the ledger. From here on the raw `litellm.completion` calls disappear from the course — every chapter calls through this door, and the end-of-notebook bill is one function call away.

In [ ]:
from shoplab.llm import llm, usage_summary, LEDGER

print(llm("In one short sentence: why do lithium batteries complicate shipping?",
          max_tokens=60))
print(LEDGER[-1])
usage_summary()

> **What you should see:** a one-sentence answer, one ledger row, and a summary with `calls: 1`. The ledger only sees traffic through `complete` — the raw calls earlier in this notebook are invisible to it, which is exactly the argument for a single door.

## Triage without hands

Parser plus wrapper is enough to attempt the chapter's real job end to end: hand the model every structured field of the ticket and demand the decision object. This time, compare against the ticket's gold label — assigned by a deterministic rules engine over the shop's actual policies (chapter 04 builds it).

In [ ]:
facts = {k: ticket[k] for k in
         ("order_id", "customer_id", "sku", "qty", "reason_text",
          "requested_action", "item_condition", "days_since_delivery",
          "evidence_photo")}
PROMPT = ("You run the ops desk at Larkspur Outfitters. Triage this return ticket:\n\n"
          + json.dumps(facts, indent=2)
          + '\n\nAnswer with one JSON object, no prose: '
            '{"decision": "...", "policy_id": "...", "refund_usd": 0.0}')

guess = parse_json_loose(llm(PROMPT, max_tokens=200))
print("model:", guess)
print("gold :", ticket["gold"])

> **What you should see:** syntactically clean JSON with confidently wrong content. The decision label need not come from Larkspur's actual vocabulary, the `policy_id` matches none of the shop's twelve real policy ids except by accident, and `refund_usd` cannot be right except by luck — the gold amount derives from the boots' price and a restocking-fee rule, and the model was shown neither.

Format was solvable with a parser. Facts are not. The model has no way to fetch order `ORD-7312`, read the restocking policy, or check the customer's tier; it can only pattern-match on what the prompt contains and improvise the rest. Giving it a way — tools, and a loop that feeds their results back — is chapter 02.

### Temperature 0 is not determinism

Every call here passed `temperature=0`, and you should still not expect byte-identical outputs across runs. Temperature 0 collapses sampling toward the highest-probability token, which shrinks variance; it does not pin down the numerics underneath. Provider-side batching, hardware differences, and silent model updates can all shift outputs between runs or between days. Treat temperature 0 as variance reduction, and treat this repo's disk cache — which replays the stored response for a byte-identical request — as the only real determinism on offer. It is also why every callout in this course states invariants, never exact strings.

## Recap

| Concept | One-liner |
|---|---|
| `ModelResponse` | The payload is `choices[0].message.content`; the receipt is `usage` plus the cost in `_hidden_params`. |
| Prompt as interface | The prompt declares the return type; ask for prose and only a human can consume the result. |
| Schema-in-prompt | An example object buys JSON-shaped output — with improvised values, and only until the model adds chatter. |
| `parse_json_loose` | Strip fences, bracket-match the first object (string-aware), `json.loads` the slice. |
| Garbage terminates | No JSON means `ValueError`, never `None` — silent defaults turn model failures into downstream lies. |
| `LEDGER` / `usage_summary()` | Every `complete()` call appends model, tokens, dollars, seconds; the totals are one call away. |
| No data, no truth | Valid JSON is not a valid decision: without access to orders and policies, the model invents both. |

## Exercises

1. Widen the contract: add a `"confidence": 0.0` field to `SCHEMA`, rerun the `PROMPT_V2` cell, and parse. Does the model comply? Try it on two or three other train tickets and study the numbers it emits — do easy and hard tickets get different confidences, or is it theater?
2. Hunt a parser gap: construct a reply that `parse_json_loose` mishandles. Start with a Python-literal object using single quotes (`{'decision': 'deny'}`), then try two JSON objects in one reply. Decide where each fix belongs — the parser, the prompt, or a retry — and sketch it.
3. Swap in the stronger model for the triage attempt: `llm(PROMPT, model=STRONG_MODEL, max_tokens=200)` — one extra call, still well under a cent. Compare the two guesses field by field. Does a stronger model land closer to gold, or does it produce the same fabrication with better manners?

**Next up:** chapter 02 gives the model hands — tools it can call, a loop that feeds results back, and a first honest shot at the right refund.